In [ ]:
import os
import json
import time
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from sklearn.decomposition import PCA
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_squared_log_error,
    r2_score,
    make_scorer
)
from sklearn.model_selection import KFold

In [ ]:
from xgboost import XGBRegressor
from skopt import BayesSearchCV
from skopt.space import Real, Integer

In [ ]:
pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 200)

In [ ]:
RANDOM_STATE = 42
USE_GPU = True  # set False to force CPU

In [ ]:
# ---- 1. Load data ----
DATA_DIR = "/kaggle/input/datasets/miftahullferdous/kickstarter-joint-embeddings/kickstarter_joint_embeddings"  # <-- update as needed

In [ ]:
TRAIN_FILE = f"{DATA_DIR}/ML dataset/ML_train.csv"
TEST_FILE = f"{DATA_DIR}/ML dataset/ML_test.csv"
BERT_FILE = f"{DATA_DIR}/NLP dataset/bert_embeddings.csv"

In [ ]:
train_df = pd.read_csv(TRAIN_FILE)
test_df = pd.read_csv(TEST_FILE)
bert_raw = pd.read_csv(BERT_FILE)

In [ ]:
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("BERT:", bert_raw.shape)

In [ ]:
# ---- 2. PCA reduce BERT embeddings (fit on train only) ----
N_COMPONENTS = 20

In [ ]:
train_ids = set(train_df["id"])
feature_cols = [c for c in bert_raw.columns if c != "id"]
train_mask = bert_raw["id"].isin(train_ids)

In [ ]:
pca = PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
pca.fit(bert_raw.loc[train_mask, feature_cols])
reduced_all = pca.transform(bert_raw[feature_cols])

In [ ]:
reduced_cols = [f"bert_pca_{i}" for i in range(N_COMPONENTS)]
reduced_df = pd.DataFrame(reduced_all, columns=reduced_cols)
reduced_df.insert(0, "id", bert_raw["id"].values)

In [ ]:
explained = pca.explained_variance_ratio_.sum()
print(f"BERT: reduced to {N_COMPONENTS} dims, explained variance = {explained:.3f}")

In [ ]:
# ---- 3. Build features ----
TARGET_RAW = "target_usd"
TARGET_LOG = "log_target"
DROP_FROM_X = ["id", "target_usd", "log_target", "goal_usd"]

In [ ]:
merged_train = train_df.merge(reduced_df, on="id", how="left")
merged_test = test_df.merge(reduced_df, on="id", how="left")

In [ ]:
assert merged_train.shape[0] == train_df.shape[0], "Row count mismatch (train)"
assert merged_test.shape[0] == test_df.shape[0], "Row count mismatch (test)"
assert merged_train[reduced_cols].isna().sum().sum() == 0
assert merged_test[reduced_cols].isna().sum().sum() == 0

In [ ]:
X_train = merged_train.drop(columns=DROP_FROM_X, errors="ignore")
X_test = merged_test.drop(columns=DROP_FROM_X, errors="ignore")
y_train = merged_train[TARGET_LOG].copy()
y_test = merged_test[TARGET_LOG].copy()
actual_usd = merged_test[TARGET_RAW].to_numpy()

In [ ]:
assert list(X_train.columns) == list(X_test.columns)

In [ ]:
# ---- 4. Evaluation ----
def evaluate_model(model_name, y_true_log, pred_log, actual_usd):
    pred_usd = np.expm1(pred_log)
    pred_usd = np.clip(pred_usd, a_min=0, a_max=None)

    mae_log = mean_absolute_error(y_true_log, pred_log)
    mse_log = mean_squared_error(y_true_log, pred_log)
    rmse_log = np.sqrt(mse_log)
    r2_log = r2_score(y_true_log, pred_log)

    mae_usd = mean_absolute_error(actual_usd, pred_usd)
    mse_usd = mean_squared_error(actual_usd, pred_usd)
    rmse_usd = np.sqrt(mse_usd)
    r2_usd = r2_score(actual_usd, pred_usd)

    rmsle = np.sqrt(mean_squared_log_error(actual_usd, pred_usd))

    return {
        "Model":      model_name,
        "MAE_log":    mae_log,
        "MSE_log":    mse_log,
        "RMSE_log":   rmse_log,
        "R2_log":     r2_log,
        "MAE_USD":    mae_usd,
        "MSE_USD":    mse_usd,
        "RMSE_USD":   rmse_usd,
        "R2_USD":     r2_usd,
        "RMSLE":      rmsle
    }, pred_usd

In [ ]:
# ---- 5. BayesSearchCV setup (XGBoost only) ----
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

In [ ]:
rmse_scorer = make_scorer(rmse, greater_is_better=False)
CV = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
N_ITER_BAYES = 4

In [ ]:
xgb_bayes_space = {
    "n_estimators":     Integer(200, 500),
    "learning_rate":    Real(0.01, 0.1, prior="log-uniform"),
    "max_depth":        Integer(4, 9),
    "min_child_weight": Integer(1, 7),
    "subsample":        Real(0.7, 1.0),
    "colsample_bytree": Real(0.7, 1.0),
    "reg_alpha":        Real(1e-3, 0.5, prior="log-uniform"),
    "reg_lambda":       Real(0.5, 3.0),
}

In [ ]:
base_estimator = XGBRegressor(
    objective="reg:squarederror", eval_metric="rmse",
    tree_method="hist", device="cuda" if USE_GPU else "cpu",
    random_state=RANDOM_STATE,
)

In [ ]:
search = BayesSearchCV(
    estimator=base_estimator, search_spaces=xgb_bayes_space, n_iter=N_ITER_BAYES,
    scoring=rmse_scorer, cv=CV, n_jobs=1,
    random_state=RANDOM_STATE, verbose=0, refit=True,
)

In [ ]:
# ---- 6. Fit + evaluate ----
print("\nTuning BERT + XGBoost via BayesSearchCV...")
start_time = time.time()
search.fit(X_train, y_train)
best_model = search.best_estimator_
tuning_time = time.time() - start_time

In [ ]:
pred_log = best_model.predict(X_test)

In [ ]:
run_label = "BERT + XGBoost (BayesSearchCV)"
metrics, _ = evaluate_model(run_label, y_test, pred_log, actual_usd)
metrics["Embedding"] = "BERT"
metrics["Base_Model"] = "XGBoost"
metrics["Tuning_Method"] = "BayesSearchCV"
metrics["Tuning_Time_Seconds"] = tuning_time
metrics["Best_CV_RMSE_log"] = -search.best_score_
metrics["Best_Params"] = dict(search.best_params_)

In [ ]:
print(f"\nBest CV RMSE_log={-search.best_score_:.4f} | "
      f"Test RMSLE={metrics['RMSLE']:.4f}, R2_log={metrics['R2_log']:.4f}, "
      f"time={tuning_time:.1f}s")

In [ ]:
result_df = pd.DataFrame([metrics])
result_df = result_df[[
    "Embedding", "Base_Model", "Tuning_Method",
    "MAE_log", "MSE_log", "RMSE_log", "R2_log",
    "MAE_USD", "MSE_USD", "RMSE_USD", "R2_USD",
    "RMSLE", "Best_CV_RMSE_log", "Tuning_Time_Seconds"
]]
print(result_df)

In [ ]:
result_df.to_csv("bert_xgb_bayes_result.csv", index=False)
with open("bert_xgb_bayes_best_params.json", "w") as f:
    json.dump(metrics["Best_Params"], f, indent=2, default=str)

In [ ]:
print("\nSaved: bert_xgb_bayes_result.csv")
print("Saved: bert_xgb_bayes_best_params.json")